# Generating Points on the Flag Manifold

## Setup

In [1]:
import torch
torch.set_default_dtype(torch.float64)
import numpy as np 
import qmcpy as qp
import agsutil
import time
import pandas as pd
import os
import matplotlib
from matplotlib import pyplot
MPLP = agsutil.mpl_setup()
agsutil.print_data_signatures(MPLP,"MPLP",verbose_indent=0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch device = %s"%DEVICE)
icdf_normal = torch.distributions.Normal(loc=torch.zeros(1,device=DEVICE),scale=torch.ones(1,device=DEVICE)).icdf

MPLP['PW'] = 30
MPLP['FS'] = 30
MPLP['COLORS'] a list of length 10
MPLP['LINESTYLES'] a list of length 10
MPLP['MARKERS'] a list of length 12
torch device = cpu


In [2]:
def tff_qr(u, lam):
    t = lam.size(-1)
    assert u.size(-1)==(t**2)
    assert (0<=u).all()
    assert (u<=1).all()
    u = u.reshape((*u.shape[:-1],t,t))
    v = icdf_normal(u)
    q,r = torch.linalg.qr(v)
    # assert torch.allclose(torch.einsum("...ij,...jk->...ik",q,r),v)
    f = torch.einsum("...ij,...j,...kj->...ik",q,lam,q)
    return f
def tff_eig(u, lam):
    t = lam.size(-1) 
    assert u.size(-1)==(t*(t+1)//2)
    alpha = icdf_normal(u[...,:t])/np.sqrt(2)
    beta = icdf_normal(u[...,t:])
    il0,il1 = torch.tril_indices(t,t,offset=-1,device=DEVICE)
    v = torch.eye(t,device=DEVICE)*alpha[...,None]
    v[...,il0,il1] = beta
    v += v.tril(-1).transpose(dim0=-2,dim1=-1)
    # assert torch.allclose(v[...,torch.arange(t,device=DEVICE),torch.arange(t,device=DEVICE)],alpha)
    gamma,q = torch.linalg.eigh(v) # TODO: flip sign of q since it is only unique up to multiplying each column by {-1,1}
    # assert torch.allclose(torch.einsum("...ij,...j,...kj->...ik",q,gamma,q),v)
    f = torch.einsum("...ij,...j,...kj->...ik",q,lam,q)
    return f
def genf_qr_iid_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.rand((n,t**2),generator=rng,device=DEVICE)
    x = tff_qr(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
def genf_qr_ld_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.from_numpy(qp.Halton(t**2,seed=seed,randomize="NUS",warn=False)(n)).to(DEVICE)
    x = tff_qr(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
def genf_eig_iid_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.rand((n,t*(t+1)//2),generator=rng,device=DEVICE)
    x = tff_eig(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
def genf_eig_ld_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.from_numpy(qp.Halton(t*(t+1)//2,seed=seed,randomize="NUS",warn=False)(n)).to(DEVICE)
    x = tff_eig(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
# genf_qr_iid_x_equal_w(n=3,lam=torch.rand(4,device=DEVICE));
# genf_qr_ld_x_equal_w(n=3,lam=torch.rand(4,device=DEVICE));
# genf_eig_iid_x_equal_w(n=3,lam=torch.rand(4,device=DEVICE));
# genf_eig_ld_x_equal_w(n=3,lam=torch.rand(4,device=DEVICE));

In [34]:
def get_q(u):
    t = int((np.sqrt(1+8*u.size(-1))-1)/2)
    z = torch.zeros((*u.shape[:-1],t,t))
    il0,il1 = torch.tril_indices(t,t,device=DEVICE)
    eyet = torch.eye(t,device=DEVICE)
    q = torch.zeros_like(z)+eyet 
    k = 0
    for i in range(t):
        v = u[...,k:(k+i+1)]
        c = (v**2).sum(-1)
        k += i+1
        q[...,:(i+1),:(i+1)] = q[...,:(i+1),:(i+1)]-2/c[...,None,None]*v[...,:,None]*(q[...,:(i+1),:(i+1)]*v[...,:,None]).sum(-2)[...,None,:]
    assert torch.allclose(torch.einsum("...ij,...kj->...ik",q,q),eyet)
    assert torch.allclose(torch.einsum("...ji,...jk->...ik",q,q),eyet)
    return q
get_q(torch.rand(4,5,3*(3+1)//2,device=DEVICE));

In [35]:
def genf_opt1_x_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    wu0 = torch.cat([
        torch.ones((n,1),device=DEVICE)/n,
        torch.from_numpy(qp.Halton(t*(t+1)//2,seed=seed,randomize="NUS",warn=False)(n)).to(DEVICE)],dim=-1)
    def loss(wu):
        w = wu[...,0]
        u = wu[...,1:]
        q = get_q(u)
        print(w.shape)
        print(u.shape) 
        assert False
    agsutil.lm_opt(
        f = loss, 
        theta0 = wu0, 
        ytrue = torch.zeros((t,t),device=DEVICE),
        iters = 5
    )

x,w = genf_opt1_x_w(n=3,lam=torch.rand(4,device=DEVICE))

RuntimeError: vmap: inplace arithmetic(self, *extra_args) is not possible because there exists a Tensor `other` in extra_args that has more elements than `self`. This happened due to `other` being vmapped over but `self` not being vmapped over in a vmap. Please try to use out-of-place operators instead of inplace arithmetic. If said operator is being called inside the PyTorch framework, please file a bug report instead.